# `avh` — AV-HuBERT features into the shared cache

Turns every clip into two fixed-length token sequences from **AV-HuBERT** (Meta, 2022), the
audio-visual speech encoder trained on LRS3 + VoxCeleb2:

```
<DATA>/cached_features/avhubert-video/<clip_id>.npy   float16  (64, 768)
<DATA>/cached_features/avhubert-audio/<clip_id>.npy   float16  (64, 768)
```

Both streams come from the **same checkpoint**: the mouth at 25 fps and the log-mel filterbank
of the same speech, each passed through the encoder in its own modality. AV-HuBERT was trained
with modality dropout, so a video-only and an audio-only pass are both in-distribution — no
other model in this project can do that.

> **Three things to know before you spend a session.** 1) The weights are under Meta's
> AV-HuBERT licence: non-commercial research, and it explicitly forbids biometric processing
> and surveillance uses — read [`docs/plan.md`](docs/plan.md). 2) It is a **fairseq** model:
> the official code targets Python 3.8, and the import may fail on a current Colab image.
> 3) There is no published pipeline for DOLOS: the mouth crop below is a faithful but
> simplified version of the official `align_mouth.py` (no mean-face stabilisation).

| | |
|---|---|
| **Time** | ~45–75 min on a T4: the alignment runs in parallel on the CPU, the GPU pass is minutes |
| **Writes** | `cached_features/avhubert-{video,audio}/` + one `manifest.json` each; crops in `/content/avhubert-crops` |
| **Reads** | `clips/`, `audio/`, `dataset.csv`, `splits/` |
| **Resumable** | yes — an existing `.npy` is skipped |

## 0 — Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
# `--recursive` matters: AV-HuBERT carries its own fairseq fork as a submodule.
!git submodule update --init --recursive --depth 1 vendor/av_hubert
import sys; sys.path.insert(0, '.')
print(sorted(os.listdir('vendor/av_hubert'))[:8])

/content/DeepLearningCourse/lie-detector
error: pathspec 'vendor/av_hubert' did not match any file(s) known to git


FileNotFoundError: [Errno 2] No such file or directory: 'vendor/av_hubert'

## 1 — Environment

This is the part that decides whether the session is worth starting. AV-HuBERT needs
**fairseq** (its bundled fork), `dlib` for the 68 landmarks, and the encoder is happiest on
the Python 3.8 the authors target. The cell below installs what it can and then *reports*
what it got; a failing import is a hard stop with an explicit message, not a traceback twenty
cells later.

In [ ]:
import subprocess
import sys

# dlib-bin ships a wheel, so pip does not spend ten minutes compiling.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dlib-bin", "soundfile",
                "python_speech_features"], check=True)

# The bundled fairseq is what the checkpoint was trained with. `--no-build-isolation` is the
# only way it installs against the pre-installed torch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
                "-e", "vendor/av_hubert/fairseq"], check=False)

import model as M
report = M.check_avhubert_environment(user_dir="vendor/av_hubert/avhubert")
print(report)
if isinstance(report["fairseq"], str) and report["fairseq"].startswith("missing"):
    raise SystemExit(
        "fairseq did not import on this Python (" + report["python"] + "). The AV-HuBERT code "
        "targets 3.8; options: (a) run this notebook on a Python 3.8 runtime, (b) install a "
        "maintained fairseq fork, (c) switch the direction to the V-JEPA 2 backend in "
        "`cross-modal-fusion/extract_vjepa2.ipynb`. See docs/plan.md."
    )

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import cv2
import dlib
import numpy as np
import torch

from lib.dataset import Clips, Protocol, cache_dir, stage_media
import model as M

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", device)

## 2 — Settings

The checkpoint is the base model pre-trained on LRS3 + VoxCeleb2. `TOKENS = 64` is the fixed
length of the cached sequence: AV-HuBERT emits 25 vectors per second, so a 19 s clip would
otherwise be 475 tokens and a 2 s clip 50.

In [ ]:
DATA = Path("/content/drive/MyDrive/dolos")

VIDEO_CACHE = "avhubert-video"
AUDIO_CACHE = "avhubert-audio"
CHECKPOINT = Path(os.environ.get("AVHUBERT_CKPT",
                                 DATA / "checkpoints/base_vox_iter5.pt"))

FPS, CROP, TOKENS = M.AVHUBERT_FPS, M.AVHUBERT_CROP, 64
FORCE = False
LIMIT = 0

# Detection and landmarks run on a copy this wide; the pixels that reach the cache still come
# from the original frame. 640 is the width the DINOv2 cache already uses.
DETECT_SIDE = 640

# The alignment is CPU work in a Python loop, the encoder is a few hundred ms of GPU: they are
# two stages (§7). `WORKERS = 0` takes every core the runtime has.
WORKERS = 0
CROPS = Path("/content/avhubert-crops")     # ~0.6 MB per clip, local disk - not Drive

print("checkpoint", CHECKPOINT, "exists" if CHECKPOINT.exists() else "MISSING")
print("video cache", cache_dir(VIDEO_CACHE, DATA))
print("audio cache", cache_dir(AUDIO_CACHE, DATA))
print("crops", CROPS, "| cores", os.cpu_count())


## 3 — Which clips

In [ ]:
folds = Protocol.load(DATA / "splits")
# The mouth frame grid and the filterbank read one mp4 and one wav per clip: stage them.
clips = Clips.load(stage_media(DATA))
print(clips.describe_source())

split_ids = []
for fold in range(1, folds.folds + 1):
    for part in folds.subject(fold):
        split_ids += part
split_ids += folds.test()
split_ids = list(dict.fromkeys(split_ids))

assert not clips.missing_from(split_ids), "the splits reference clips absent from dataset.csv"

ids = clips.with_media(split_ids)
# Both modalities are needed now: the mouth crop comes from the clip, the filterbank from the
# wav the preprocessing wrote next to it.
no_audio = [clip_id for clip_id in ids if clips[clip_id].audio is None]
if no_audio:
    print(f"{len(no_audio)} clips have no wav and are left out (first {no_audio[:3]})")
    ids = [clip_id for clip_id in ids if clip_id not in set(no_audio)]
print(f"protocol {len(split_ids)} clips -> {len(ids)} with video and audio")
print(clips.describe(ids))

## 4 — Mouth ROI at 25 fps

AV-HuBERT was trained on mouth crops aligned to a mean face. This is the simplified version:

1. decode the clip and keep **25 frames per second**, one frame at a time — a 19 s clip is 475
   full-resolution frames, and holding them all costs hundreds of megabytes for nothing;
2. detect the face with dlib's HOG detector **once per second** and predict 68 landmarks on
   every frame inside the last known box — one detection per second, not one per frame;
3. take the mouth landmarks (points 48–67), expand the box to include the chin and the corners,
   square it, resize to 96×96 and convert to grayscale.

Steps 2 and 3 run on a copy `DETECT_SIDE` pixels wide and only the box comes back to the
original frame: the detector's cost grows with the area it is handed, and the landmarks only
have to be right to a couple of pixels in a 96×96 crop. That is where the time of this
notebook goes — `dlib` is CPU code, and the GPU stays idle until §7.

No mean-face stabilisation: the crop follows the head instead of removing its rotation. That
is a real difference from the paper's preprocessing and it is recorded in the manifest.


In [ ]:
# keep the landmark model next to the checkpoint, never inside the submodule
predictor_path = CHECKPOINT.parent / "shape_predictor_68_face_landmarks.dat"
predictor_path.parent.mkdir(parents=True, exist_ok=True)
if not predictor_path.exists():
    import urllib.request
    urllib.request.urlretrieve(
        "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2",
        predictor_path.with_suffix(".dat.bz2"),
    )
    import bz2
    with bz2.open(predictor_path.with_suffix(".dat.bz2"), "rb") as source, \
            open(predictor_path, "wb") as target:
        target.write(source.read())

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(str(predictor_path))

# One thread per process: the parallelism is across clips (§7), and OpenCV's own pool would
# only fight it for the same cores.
cv2.setNumThreads(1)


def iter_25fps(path):
    """The clip resampled to 25 fps, yielded one BGR frame at a time."""
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"cannot open {path}")
    try:
        source_fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
        step, index, take = source_fps / FPS, 0, 0.0
        ok, frame = capture.read()
        while ok:
            if index >= round(take):
                yield frame
                take += step
            ok, frame = capture.read()
            index += 1
    finally:
        capture.release()


def mouth_crop(frame, box=None):
    """96x96 grayscale mouth ROI; `box` is the last detected face, in `small` coordinates."""
    height, width = frame.shape[:2]
    small = cv2.resize(frame, (DETECT_SIDE, max(1, round(DETECT_SIDE * height / width))),
                       interpolation=cv2.INTER_AREA)
    small = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    if box is None:
        faces = detector(small, 0)
        box = faces[0] if faces else None
    if box is None:                                   # no face: centre crop, keep the shape
        side = min(height, width)
        top, left = (height - side) // 2, (width - side) // 2
        patch = cv2.cvtColor(frame[top:top + side, left:left + side], cv2.COLOR_BGR2GRAY)
        return cv2.resize(patch, (CROP, CROP)), None

    points = predictor(small, box)
    mouth = np.array([(points.part(i).x, points.part(i).y) for i in range(48, 68)])
    x0, y0 = mouth.min(axis=0)
    x1, y1 = mouth.max(axis=0)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max((x1 - x0) * 1.6, (y1 - y0) * 2.2)      # chin and corners
    scale_x, scale_y = width / small.shape[1], height / small.shape[0]
    left, top = int(max(cx - side / 2, 0) * scale_x), int(max(cy - side / 2, 0) * scale_y)
    right = int(min(cx + side / 2, small.shape[1]) * scale_x)
    bottom = int(min(cy + side / 2, small.shape[0]) * scale_y)
    patch = frame[top:bottom, left:right]
    if patch.size == 0:
        patch = frame
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    return cv2.resize(gray, (CROP, CROP), interpolation=cv2.INTER_AREA), box


def mouth_sequence(path):
    """(T, 96, 96) uint8 mouth crops at 25 fps — what the GPU pass reads back."""
    crops, box = [], None
    for position, frame in enumerate(iter_25fps(path)):
        if position % FPS == 0:                       # re-detect once per second
            box = None
        crop, box = mouth_crop(frame, box)
        crops.append(crop)
    if not crops:
        raise RuntimeError(f"no frame decoded from {path}")
    return np.stack(crops)


def video_tensor(crops):
    """(T, 96, 96) uint8 -> (1, T, 1, 96, 96) float in [0, 1]."""
    return torch.from_numpy(crops).float().div_(255.0).unsqueeze(1).unsqueeze(0).contiguous()


In [ ]:
def audio_tensor(path, frames):
    """The same speech as the model expects it: 26 log-mel bins, 10 ms hop, stacked 4.

    `M.audio_features` reproduces `hubert_dataset.py`: logfbank, stack four frames, then a
    per-frame layer norm over the 104 bins. The result is 25 vectors/s, one per video frame,
    and is padded or trimmed to `frames` so the two modalities stay aligned.
    """
    features = M.audio_features(path)                 # (T, 104) at 25 fps
    if features.shape[0] >= frames:
        features = features[:frames]
    else:
        padding = torch.zeros(frames - features.shape[0], features.shape[1])
        features = torch.cat([features, padding], dim=0)
    return features.unsqueeze(0).contiguous()         # (1, T, 104)

## 5 — The encoder

`load_avhubert` reads the fairseq checkpoint, forces the video-only modality (the audio
branch is redundant next to WavLM) and freezes everything. The features are 25 fps vectors of
width 768 for the base model; `pool_avhubert` brings them to `TOKENS`.

In [ ]:
encoder, WIDTH = M.load_avhubert(CHECKPOINT, device=str(device),
                                 modalities=("video", "audio"))
print(f"AV-HuBERT | width {WIDTH} | modalities {encoder.modalities}")

clip_id = ids[0]
probe_video = video_tensor(mouth_sequence(clips[clip_id].video))   # one clip, aligned once
probe_audio = audio_tensor(clips[clip_id].audio, probe_video.shape[1])
print("probe", tuple(probe_video.shape), tuple(probe_audio.shape))

encoder.modalities = ["video"]
with torch.inference_mode():
    video_features = M.encode_avhubert(encoder, video=probe_video.to(device))
encoder.modalities = ["audio"]
with torch.inference_mode():
    audio_features = M.encode_avhubert(encoder, audio=probe_audio.to(device))
print("video", tuple(video_features.shape), "->", tuple(M.pool_avhubert(video_features, TOKENS).shape))
print("audio", tuple(audio_features.shape), "->", tuple(M.pool_avhubert(audio_features, TOKENS).shape))


## 6 — The cache

In [ ]:
def settings(name, modality):
    return {
        "encoder": name, "checkpoint": CHECKPOINT.name, "weights": "pretrained",
        "modality": modality, "fps": FPS, "crop": CROP if modality == "video" else None,
        "tokens": TOKENS, "width": WIDTH, "detect_side": DETECT_SIDE,
        "alignment": f"dlib68_mouth_bbox_no_meanface_detect{DETECT_SIDE}",
        "fbank": [M.FBANK_BINS, M.FBANK_STACK] if modality == "audio" else None,
        "dtype": "float16",
    }


## 7 — Two stages: align on the CPU, encode on the GPU

The alignment is a Python loop over 25 frames per second of video and the encoder is a few
hundred milliseconds of GPU. Run them in one loop and the GPU waits 95% of the time — which is
what a single-stage version does. They are therefore split:

1. **`build_crops`** — every worker takes a clip, decodes it, aligns it, writes `(T, 96, 96)`
   uint8 to `CROPS`. Resumable per clip, so a disconnect costs one clip; `WORKERS = 0` uses
   every core the runtime has. The workers never touch CUDA: the GPU belongs to the parent.
2. **`encode_all`** — reads the crops back and runs the two passes. No decoding, no `dlib`, no
   Python loop over frames, so the GPU is fed back to back.

The crop folder is the seam: re-running `encode_all` (a few minutes) after changing `TOKENS` or
the pooling costs nothing, while `build_crops` (the expensive half) is already done.


In [ ]:
import multiprocessing

from tqdm import tqdm     # the console bar: it renders in Colab even when the widget does not


def align_clip(clip_id):
    """One clip: decode, align, write the crops. Runs in a worker process."""
    try:
        crops = mouth_sequence(clips[clip_id].video)
        temporary = CROPS / f".{clip_id}.tmp.npy"
        np.save(temporary, crops)
        temporary.rename(CROPS / f"{clip_id}.npy")
        return clip_id, len(crops), None
    except Exception as error:                        # one clip, not the run
        return clip_id, 0, f"{type(error).__name__}: {error}"[:200]


def build_crops(clip_ids, limit=0):
    CROPS.mkdir(parents=True, exist_ok=True)
    for stale in CROPS.glob(".*.tmp.npy"):
        stale.unlink()
    todo = [c for c in clip_ids if FORCE or not (CROPS / f"{c}.npy").exists()]
    if limit:
        todo = todo[:limit]
    workers = WORKERS or os.cpu_count() or 1
    print(f"{len(todo)} clips to align ({len(clip_ids) - len(todo)} already there), "
          f"{workers} workers")
    if not todo:
        return {}

    started, failures = time.perf_counter(), {}
    # fork, not spawn: workers inherit `clips`, `detector` and `CROPS` instead of
    # re-importing the notebook. Linux only, which is what Colab gives us.
    with multiprocessing.get_context("fork").Pool(workers) as pool:
        for position, (clip_id, frames, error) in enumerate(
                pool.imap_unordered(align_clip, todo, chunksize=4), 1):
            if error:
                failures[clip_id] = error
                print(f"  {clip_id}: {error[:80]}")
            if position % 25 == 0 or position == len(todo):
                rate = position / (time.perf_counter() - started)
                print(f"  {position}/{len(todo)}  {rate:.2f} clips/s  "
                      f"eta {(len(todo) - position) / rate / 60:.0f} min")

    on_disk = sum(path.stat().st_size for path in CROPS.glob("*.npy")) / 1e6
    print(f"aligned {len(todo) - len(failures)} clips in "
          f"{time.perf_counter() - started:.0f} s ({on_disk:.0f} MB in {CROPS}), "
          f"{len(failures)} failed")
    return failures


def encode_all(clip_ids, limit=0):
    pending = set(cache_v.pending(clip_ids)) | set(cache_a.pending(clip_ids))
    todo = [c for c in clip_ids if c in pending and (CROPS / f"{c}.npy").exists()]
    missing = [c for c in clip_ids if c in pending and not (CROPS / f"{c}.npy").exists()]
    if missing:
        print(f"{len(missing)} cached clips have no crops yet — build_crops first "
              f"(first {missing[:3]})")
    if limit:
        todo = todo[:limit]
    if not todo:
        print("nothing to encode")
        return
    print(f"{len(todo)} clips to encode ({len(clip_ids) - len(todo)} already cached)")

    started = time.perf_counter()
    progress = tqdm(todo, desc="encoding", unit="clip", dynamic_ncols=True)
    for position, clip_id in enumerate(progress, 1):
        try:
            video = video_tensor(np.load(CROPS / f"{clip_id}.npy")).to(device)
            audio = audio_tensor(clips[clip_id].audio, video.shape[1]).to(device)

            encoder.modalities = ["video"]
            with torch.inference_mode():
                features_v = M.encode_avhubert(encoder, video=video)
            cache_v.write(clip_id, M.pool_avhubert(features_v, TOKENS)[0].float().cpu().numpy())

            encoder.modalities = ["audio"]
            with torch.inference_mode():
                features_a = M.encode_avhubert(encoder, audio=audio)
            cache_a.write(clip_id, M.pool_avhubert(features_a, TOKENS)[0].float().cpu().numpy())

            cache_v.meta["failures"].pop(clip_id, None)
            cache_a.meta["failures"].pop(clip_id, None)
        except Exception as error:                     # one clip, not the run
            message = f"{type(error).__name__}: {str(error)[:180]}"
            cache_v.meta["failures"][clip_id] = message
            cache_a.meta["failures"][clip_id] = message
            progress.write(f"  {clip_id}: {message[:80]}")   # above the bar, not through it

        rate = position / (time.perf_counter() - started)
        progress.set_postfix_str(f"{rate:.2f} clips/s, {len(cache_v.meta['failures'])} failed")
        if position % 25 == 0 or position == len(todo):
            cache_v.save(); cache_a.save()   # the manifest is what makes an interrupted run resumable

    for cache in (cache_v, cache_a):
        cache.meta["count"] = len(list(cache.dir.glob("*.npy")))
        cache.save()
    print("cached", cache_v.meta["count"], "video /", cache_a.meta["count"], "audio arrays")


build_crops(ids[:8])        # enough to see the rate and check the crop in §8
encode_all(ids[:8])


## 8 — Look at one clip

Twenty-five frames of the same mouth, on a grid. If the crop drifts off the mouth, stop here:
the encoder would be reading the chin.

In [ ]:
import matplotlib.pyplot as plt

clip_id = ids[0]
path = CROPS / f"{clip_id}.npy"
crops = np.load(path) if path.exists() else mouth_sequence(clips[clip_id].video)

figure, axes = plt.subplots(4, 4, figsize=(8, 8))
for axis, frame in zip(axes.ravel(), crops[::5]):
    axis.imshow(frame, cmap="gray"); axis.axis("off")
figure.suptitle(f"{clip_id} — mouth ROI at 25 fps, {clips[clip_id].label}", fontsize=10)
plt.tight_layout(); plt.show()

stored = np.load(cache_dir(VIDEO_CACHE, DATA) / f"{clip_id}.npy")
print(f"{clip_id}: crops {crops.shape} | stored {stored.shape} {stored.dtype} "
      f"| finite {np.isfinite(stored).all()} | std {stored.std():.3f}")


## 9 — Full run

In [ ]:
encode_all(ids)

## 10 — Verify

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

for name in (VIDEO_CACHE, AUDIO_CACHE):
    directory = cache_dir(name, DATA)
    files = sorted(directory.glob("*.npy"))
    size_mb = sum(path.stat().st_size for path in files) / 1e6
    sample = np.load(files[0]) if files else np.zeros((0,))
    absent = [clip_id for clip_id in ids if not (directory / f"{clip_id}.npy").exists()]
    print(f"{name:16} {len(files):5d} files  {size_mb:7.1f} MB  shape {sample.shape}  "
          f"missing {len(absent)}")

directory = cache_dir(AUDIO_CACHE, DATA)
train_ids, val_ids, _ = folds.subject(1)
train_ids = [c for c in train_ids if (directory / f"{c}.npy").exists()]
val_ids = [c for c in val_ids if (directory / f"{c}.npy").exists()]
if train_ids and val_ids:
    x_train = np.stack([np.load(directory / f"{c}.npy").mean(0) for c in train_ids])
    x_val = np.stack([np.load(directory / f"{c}.npy").mean(0) for c in val_ids])
    probe_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    probe_model.fit(x_train, clips.labels(train_ids))
    print(f"audio-only probe on fold 1 val: "
          f"AUC {roc_auc_score(clips.labels(val_ids), probe_model.predict_proba(x_val)[:, 1]):.3f}")
print("(plumbing check — the number that counts comes from the five folds in train.ipynb)")

## 11 — Next

`train.ipynb` reads `avhubert-video` + `avhubert-audio` by default. Both streams are 768 wide
and both come from the same encoder, so `unified` fuses them outside the model and
`congruence` compares two representations that were trained to agree — the cleanest version of
the incongruence test this project has.

The model's own fused pass (`modalities = ["audio", "video"]`, one forward for both) is not
cached: it emits a single sequence, which would make a two-stream probe impossible. If you
want that baseline, add a third cache with `modality="av"` and a `fused` variant.